In [1]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image


C:\Users\0871\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class EnsembleModel(nn.Module):
    def __init__(self, num_classes):
        super(EnsembleModel, self).__init__()
        self.resnet50 = models.resnet50(pretrained=True)
        self.resnet50.fc = nn.Identity()  # Output: 2048 features
        
        self.mobilenetv2 = models.mobilenet_v2(pretrained=True)
        self.mobilenetv2.classifier[1] = nn.Identity()  # Output: 1280 features
        
        self.alexnet = models.alexnet(pretrained=True)
        self.alexnet.classifier[6] = nn.Identity()  # Output: 4096 features
        
        # Freeze the pre-trained model parameters
        for param in self.resnet50.parameters():
            param.requires_grad = False
        for param in self.mobilenetv2.parameters():
            param.requires_grad = False
        for param in self.alexnet.parameters():
            param.requires_grad = False
        
        # Adjust the classifier to match the combined feature size: 7424
        self.classifier = nn.Linear(7424, num_classes)

    def forward(self, x):
        resnet_features = self.resnet50(x)
        mobilenet_features = self.mobilenetv2(x)
        alexnet_features = self.alexnet(x)
        
        # Concatenate features
        combined_features = torch.cat((resnet_features, mobilenet_features, alexnet_features), dim=1)
        
        # Pass concatenated features through the custom classifier
        output = self.classifier(combined_features)
        return output


In [3]:
# Initialize the model and load the saved weights
num_classes = 5  # This should match the number of classes used during training
ensemble_model = EnsembleModel(num_classes=num_classes)
ensemble_model.load_state_dict(torch.load('ensemble_model.pth'))
ensemble_model.eval()  # Set the model to evaluation mode


C:\Users\0871\AppData\Roaming\Python\Python310\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\0871\AppData\Roaming\Python\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\0871\AppData\Roaming\Python\Python310\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use

EnsembleModel(
  (resnet50): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
   

In [8]:
image_path = r'data\02-Mild\9002116R.png'  # Update this path to your image
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

image = Image.open(image_path).convert('RGB')
image = data_transforms(image)
image = image.unsqueeze(0)  # Add a batch dimension


In [9]:
# Assuming the classes are 'Normal', 'Doubtful', 'Mild', 'Moderate', 'Severe'
class_names = ['Normal', 'Doubtful', 'Mild', 'Moderate', 'Severe']


In [10]:
with torch.no_grad():
    outputs = ensemble_model(image)
    _, predicted = torch.max(outputs, 1)
    predicted_class = class_names[predicted.item()]
    print(f'Predicted class: {predicted_class}')


Predicted class: Mild
